In [92]:
import torch



In [93]:
DATA_PATH = "data/tinyshakespeare.txt"

In [94]:
with (open(DATA_PATH,"r",encoding="utf-8") as f):
    text = f.read()
len(text)

1115389

In [95]:
print(text[:1000])

First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You are all resolved rather to die than to famish?

All:
Resolved. resolved.

First Citizen:
First, you know Caius Marcius is chief enemy to the people.

All:
We know't, we know't.

First Citizen:
Let us kill him, and we'll have corn at our own price.
Is't a verdict?

All:
No more talking on't; let it be done: away, away!

Second Citizen:
One word, good citizens.

First Citizen:
We are accounted poor citizens, the patricians good.
What authority surfeits on would relieve us: if they
would yield us but the superfluity, while it were
wholesome, we might guess they relieved us humanely;
but they think we are too dear: the leanness that
afflicts us, the object of our misery, is as an
inventory to particularise their abundance; our
sufferance is a gain to them Let us revenge this with
our pikes, ere we become rakes: for the gods know I
speak this in hunger for bread, not in thirst for revenge.



In [96]:
chars = sorted(list(set(text)))
vocab_size = len(chars)
print("".join(chars))
print(vocab_size)


 !$&',-.3:;?ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz
65


In [97]:
char_to_int = {ch:i for i,ch in enumerate(chars)}
int_to_char = {i:ch for i,ch in enumerate(chars)}

In [98]:
encode = lambda s : [char_to_int[e] for e in s]
decode = lambda i : "".join([int_to_char[e] for e in i])

In [99]:
encode_data = encode(text)
data = torch.tensor(encode_data,dtype=torch.long)
print(data.shape,data.dtype)
print(data[:100])

torch.Size([1115389]) torch.int64
tensor([18, 47, 56, 57, 58,  1, 15, 47, 58, 47, 64, 43, 52, 10,  0, 14, 43, 44,
        53, 56, 43,  1, 61, 43,  1, 54, 56, 53, 41, 43, 43, 42,  1, 39, 52, 63,
         1, 44, 59, 56, 58, 46, 43, 56,  6,  1, 46, 43, 39, 56,  1, 51, 43,  1,
        57, 54, 43, 39, 49,  8,  0,  0, 13, 50, 50, 10,  0, 31, 54, 43, 39, 49,
         6,  1, 57, 54, 43, 39, 49,  8,  0,  0, 18, 47, 56, 57, 58,  1, 15, 47,
        58, 47, 64, 43, 52, 10,  0, 37, 53, 59])


In [100]:
train_size = 0.9
n = int(len(data)*train_size)
train_data = data[:n]
test_data = data[n:]
train_data.shape,test_data.shape

(torch.Size([1003850]), torch.Size([111539]))

In [101]:
block_size = 8
train_data[:block_size+1]

tensor([18, 47, 56, 57, 58,  1, 15, 47, 58])

In [102]:
x = train_data[:block_size]
y = train_data[1:block_size+1]

for t in range(block_size):
    context = x[:t+1]
    target = y [t]
    print(f"context: {context}       target:{target}")

context: tensor([18])       target:47
context: tensor([18, 47])       target:56
context: tensor([18, 47, 56])       target:57
context: tensor([18, 47, 56, 57])       target:58
context: tensor([18, 47, 56, 57, 58])       target:1
context: tensor([18, 47, 56, 57, 58,  1])       target:15
context: tensor([18, 47, 56, 57, 58,  1, 15])       target:47
context: tensor([18, 47, 56, 57, 58,  1, 15, 47])       target:58


In [103]:
batch_size = 4
block_size = 8

def get_batch(data):
    ix = torch.randint(len(data)- block_size,(batch_size,))
    x = torch.stack([data[i:i+block_size] for i in ix])
    y = torch.stack([data[i+1:i+block_size+1] for i in ix])
    return x,y

xb,yb = get_batch(train_data)
print("inputs:")
print(xb.shape)
print(xb)
print("targets:")
print(yb.shape)
print(yb)



    

inputs:
torch.Size([4, 8])
tensor([[43, 57, 58,  1, 44, 43, 50, 50],
        [43, 52,  0, 32, 53,  1, 58, 59],
        [50,  1, 51, 39, 49, 43,  1, 45],
        [ 1, 21,  6,  1, 61, 46, 53,  1]])
targets:
torch.Size([4, 8])
tensor([[57, 58,  1, 44, 43, 50, 50, 53],
        [52,  0, 32, 53,  1, 58, 59, 56],
        [ 1, 51, 39, 49, 43,  1, 45, 53],
        [21,  6,  1, 61, 46, 53,  1, 39]])


In [104]:
for b in range(batch_size):
    for t in range(block_size):
        context = xb[b, :t+1]
        target = yb [b,t]
        print(f"context: {context}       target:{target}")
    print("-------")

context: tensor([43])       target:57
context: tensor([43, 57])       target:58
context: tensor([43, 57, 58])       target:1
context: tensor([43, 57, 58,  1])       target:44
context: tensor([43, 57, 58,  1, 44])       target:43
context: tensor([43, 57, 58,  1, 44, 43])       target:50
context: tensor([43, 57, 58,  1, 44, 43, 50])       target:50
context: tensor([43, 57, 58,  1, 44, 43, 50, 50])       target:53
-------
context: tensor([43])       target:52
context: tensor([43, 52])       target:0
context: tensor([43, 52,  0])       target:32
context: tensor([43, 52,  0, 32])       target:53
context: tensor([43, 52,  0, 32, 53])       target:1
context: tensor([43, 52,  0, 32, 53,  1])       target:58
context: tensor([43, 52,  0, 32, 53,  1, 58])       target:59
context: tensor([43, 52,  0, 32, 53,  1, 58, 59])       target:56
-------
context: tensor([50])       target:1
context: tensor([50,  1])       target:51
context: tensor([50,  1, 51])       target:39
context: tensor([50,  1, 51, 3

In [105]:
import torch
from torch import nn
from torch.nn import functional as F

class BigramLanguageModel(nn.Module):
    def __init__(self,vocab_size):
        super().__init__()
        self.token_embedding = nn.Embedding(vocab_size,vocab_size)
        
    def forward(self,idx,targets=None):
        #idx , targets are (B,T)
        out = self.token_embedding(idx) # (batch,token,channels)
        
        if targets is None:
            loss = None
        else:
            B,T,C = out.shape
            out = out.view(B*T,C)
            targets = targets.view(B*T)
            loss = F.cross_entropy(out,targets)
        return out,loss
    
    def generate(self,idx,max_new_token):
        for _ in range(max_new_token):
            logits, loss = self(idx) # (B,T,C) ex (4,8,65)
            
            logits = logits[:,-1,:] # => (B,C)
            probs = F.softmax(logits,dim=-1)
            idx_next = torch.multinomial(probs,num_samples=1) # (B,1)
            idx = torch.cat((idx,idx_next),dim=1) # (B,T+1)
        return idx
            
    
m = BigramLanguageModel(vocab_size)
out,loss = m(xb,yb)
print(out.shape)
print(loss)

torch.Size([32, 65])
tensor(4.7416, grad_fn=<NllLossBackward0>)


In [106]:
idx = torch.zeros((1,1),dtype=torch.long)
result = decode(m.generate(idx,max_new_token=100)[0].tolist())
print(result)


ZAgrrx.dB.tkcwEBGdU3poPeN3PJ-3?'H;rol!pKl:ItkHIH$&L?cNlUPmw NtErKcfjmKZ?FDBvbzxKek
HSfQJs,cKESEi;Wqz


In [107]:
opt = torch.optim.AdamW(m.parameters(),lr=1e-3)

In [108]:
batch_size = 32

for step in range(10_000):
    xb,yb = get_batch(train_data)
    logits , loss = m(xb,yb)
    opt.zero_grad(set_to_none=True)
    loss.backward()
    opt.step()
print(loss.item())

2.47273325920105


In [109]:
idx = torch.zeros((1,1),dtype=torch.long)
result = decode(m.generate(idx,max_new_token=300)[0].tolist())
print(result)


I a!
S:
ARORKUS:
Whthishou.

Bure thiord oche ty Xzsheame cispr mit ot as thail goutheandd d yo y be Whe sshary ncif NGhevera f ho IAng nge h'doursoresen plyoowe brangrerlat lyof mal WANUCherourun thoriotes hathere JKIUCXMAUndeelmY:
GIAnds.
Was n:

K:
Hg ooshe s:
An, G grefre t, can eanory'e, ay, ho


# maths trick in self-attention

In [110]:
torch.manual_seed(1337)
B,T,C = 4,8,2 # batch, time, channels
x = torch.randn(B,T,C)
x.shape
# average on all previous tokens 

torch.Size([4, 8, 2])

In [111]:
xbow = torch.zeros((B,T,C))
for b in range(B):
    for t in range(T):
        xprev = x[b,:t+1]
        xbow[b,t] = torch.mean(xprev,dim=0)
        #print(xprev.shape)
print(xbow.shape)

torch.Size([4, 8, 2])


In [112]:
# can't comminucate with the past with masks
tril = torch.tril(torch.ones(T,T))
wei = torch.zeros((T,T))
wei = wei.masked_fill(tril == 0,float("-inf"))
wei = F.softmax(wei,dim=-1)
xbow3 = wei @ x
wei

tensor([[1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.5000, 0.5000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.3333, 0.3333, 0.3333, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.2500, 0.2500, 0.2500, 0.2500, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.2000, 0.2000, 0.2000, 0.2000, 0.2000, 0.0000, 0.0000, 0.0000],
        [0.1667, 0.1667, 0.1667, 0.1667, 0.1667, 0.1667, 0.0000, 0.0000],
        [0.1429, 0.1429, 0.1429, 0.1429, 0.1429, 0.1429, 0.1429, 0.0000],
        [0.1250, 0.1250, 0.1250, 0.1250, 0.1250, 0.1250, 0.1250, 0.1250]])

In [113]:
B,T,C = 2,4,2

embed = torch.randn(B,T,C)
pos = torch.randn(T,C)
x = embed+pos

embed.shape,pos.shape,x.shape

(torch.Size([2, 4, 2]), torch.Size([4, 2]), torch.Size([2, 4, 2]))

In [114]:
embed

tensor([[[-0.8345,  0.5978],
         [-0.0514, -0.0646],
         [-0.4970,  0.4658],
         [-0.2573, -1.0673]],

        [[ 2.0089, -0.5370],
         [ 0.2228,  0.6971],
         [-1.4267,  0.9059],
         [ 0.1446,  0.2280]]])

In [115]:
pos

tensor([[-0.8437, -1.3370],
        [ 0.5897, -0.2279],
        [ 0.5801,  1.1431],
        [-0.7992,  1.5127]])

In [116]:
x

tensor([[[-1.6782, -0.7392],
         [ 0.5383, -0.2924],
         [ 0.0831,  1.6089],
         [-1.0564,  0.4455]],

        [[ 1.1652, -1.8740],
         [ 0.8125,  0.4692],
         [-0.8465,  2.0491],
         [-0.6546,  1.7407]]])

In [117]:
# version 4 : self-attention
B,T,C = 2,4,2
x = torch.randn(B,T,C)

tril = torch.tril(torch.ones(T,T))
wei = torch.zeros((T,T))
wei = wei.masked_fill(tril==0,float('-inf'))
wei = F.softmax(wei,dim=1)
out = wei @ x 

x.shape,wei.shape,out.shape

(torch.Size([2, 4, 2]), torch.Size([4, 4]), torch.Size([2, 4, 2]))

In [118]:
x

tensor([[[-1.6669, -1.3651],
         [-0.1655,  0.9623],
         [ 0.0315, -0.7419],
         [-0.2978,  0.0172]],

        [[-0.1772, -0.1334],
         [ 0.2940,  1.3850],
         [ 0.1209,  2.5418],
         [-0.6405, -1.9740]]])

In [119]:
wei

tensor([[1.0000, 0.0000, 0.0000, 0.0000],
        [0.5000, 0.5000, 0.0000, 0.0000],
        [0.3333, 0.3333, 0.3333, 0.0000],
        [0.2500, 0.2500, 0.2500, 0.2500]])

In [120]:
out

tensor([[[-1.6669, -1.3651],
         [-0.9162, -0.2014],
         [-0.6003, -0.3816],
         [-0.5247, -0.2819]],

        [[-0.1772, -0.1334],
         [ 0.0584,  0.6258],
         [ 0.0792,  1.2644],
         [-0.1007,  0.4548]]])

In [121]:
torch.manual_seed(1337)
B,T,C = 4,8,32
x = torch.randn(B,T,C)

head_size = 16
key_layer = nn.Linear(C,head_size,bias = False)
query_layer = nn.Linear(C,head_size,bias=False)
value_layer = nn.Linear(C,head_size,bias=False)

k = key_layer(x) # (B,T,16)
q = query_layer(x) # (B,T,16)

wei = q @ k.transpose(-2,-1) # (B,T,16) @ (B,16,T) => (B,T,T)

tril = torch.tril(torch.ones(T,T))
wei = wei.masked_fill(tril==0,float('-inf'))
wei = F.softmax(wei,dim=-1)

v = value_layer(x)
out = wei @ v

out.shape

torch.Size([4, 8, 16])

In [122]:
wei.shape,wei[0]

(torch.Size([4, 8, 8]),
 tensor([[1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
         [0.1574, 0.8426, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
         [0.2088, 0.1646, 0.6266, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
         [0.5792, 0.1187, 0.1889, 0.1131, 0.0000, 0.0000, 0.0000, 0.0000],
         [0.0294, 0.1052, 0.0469, 0.0276, 0.7909, 0.0000, 0.0000, 0.0000],
         [0.0176, 0.2689, 0.0215, 0.0089, 0.6812, 0.0019, 0.0000, 0.0000],
         [0.1691, 0.4066, 0.0438, 0.0416, 0.1048, 0.2012, 0.0329, 0.0000],
         [0.0210, 0.0843, 0.0555, 0.2297, 0.0573, 0.0709, 0.2423, 0.2391]],
        grad_fn=<SelectBackward0>))

In [123]:
k.shape,k.transpose(-2,-1).shape

(torch.Size([4, 8, 16]), torch.Size([4, 16, 8]))

In [124]:
k = torch.randn(B,T,head_size)
q = torch.randn(B,T,head_size)
wei = q @ k.transpose(-2,-1)  * head_size**-0.5 # control the variance
k.var(),q.var(),wei.var()

(tensor(1.0449), tensor(1.0700), tensor(1.0918))

SyntaxError: can't use starred expression here (794582372.py, line 1)